#🛠️ 0. Install MCP dependencies

In [1]:
!pip -q install fastmcp nest_asyncio openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.5/367.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.3/184.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-auth 2.38.0 requires cachetools<6.0,>=2.0.0, but you have cachetools 6.2.1 which is incompatible.


In [3]:
import os, json, asyncio, nest_asyncio, textwrap, uuid
from typing import Any, Dict, List, Optional
from fastmcp import Client as MCPClient
from openai import OpenAI
from google.colab import userdata

nest_asyncio.apply()

# ---- CONFIG ----
SERVER_URL = os.getenv("SERVER_URL", "https://thalhathai-mcp-rag-tutorial.hf.space/mcp")  # include /mcp
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')  # set in Colab: %env OPENAI_API_KEY=sk-...
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")

if not OPENAI_API_KEY:
    raise ValueError("Set OPENAI_API_KEY env var before running.")

oai = OpenAI(api_key=OPENAI_API_KEY)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
def pjson(x):
    try:
        print(json.dumps(x, indent=2, ensure_ascii=False))
    except Exception:
        print(x)

def unwrap(res):
    """Return bare dict from a fastmcp response."""
    return getattr(res, "data", res)

async def mcp_connect(url: str) -> MCPClient:
    if not url or "<your-hf-space>" in url:
        raise ValueError("Set SERVER_URL to your deployed MCP endpoint (…/mcp).")
    client = MCPClient(url)
    await client.__aenter__()
    await client.ping()
    return client

async def mcp_close(client: MCPClient):
    try:
        await client.__aexit__(None, None, None)
    except Exception:
        pass


In [5]:
TOOL_SPECS = {
    "add": {
        "desc": "Add two integers and return their sum.",
        "args": {"a": "int", "b": "int"}
    },
    "list_students": {
        "desc": "List all known student names.",
        "args": {}
    },
    "get_student_metadata": {
        "desc": "Return full metadata for a student by name (case-insensitive).",
        "args": {"name": "str"}
    },
    "get_student_email": {
        "desc": "Return the email for a student by name (case-insensitive).",
        "args": {"name": "str"}
    },
    "search_student_by_field": {
        "desc": "Case-insensitive contains() search across a metadata field (e.g., department='Computer').",
        "args": {"field": "str", "value": "str"}
    },
    "reload_metadata": {
        "desc": "Reload metadata files from disk.",
        "args": {}
    },
    "search_rag": {
        "desc": "Semantic search via Pinecone with OpenAI embeddings; returns top_k matches.",
        "args": {
            "query": "str",
            "top_k": "int",
            "namespace": "Optional[str]"
        }
    },
    "find_candidates": {
        "desc": (
            "Find N student candidates for one or more topics. "
            "Tries metadata fields first (research_interests/skills/keywords/department/project_title/summary), "
            "then falls back to RAG per topic. Aggregates by student with evidence and can enforce distinct projects."
        ),
        "args": {
            "topics": "Union[str, List[str]]",
            "n": "int (default=2)",
            "prefer_fields": "Optional[List[str]]",
            "ensure_distinct_projects": "bool (default=True)",
            "top_k_per_topic": "int (default=5)",
            "namespace": "Optional[str]"
        }
    }
}


In [6]:
{"action":"tool","name":"search_student_by_field","args":{"field":"department","value":"Computer"}}


{'action': 'tool',
 'name': 'search_student_by_field',
 'args': {'field': 'department', 'value': 'Computer'}}

In [7]:
PLANNER_SYS = """You are a concise assistant that can call tools on a university project metadata + RAG server (MCP).
You MUST either:
- propose exactly ONE tool call with arguments, OR
- produce the final answer.

Output STRICT JSON with keys:
- action: "tool" or "final"
- If action="tool": include "name" and "args" exactly as a JSON object
- If action="final": include "answer" (string)
Do not include extra keys. No markdown. No prose outside JSON.
Use tools when needed instead of guessing.
"""

def tool_catalog_markdown():
    lines = []
    for k, spec in TOOL_SPECS.items():
        lines.append(f"- {k}: {spec['desc']} args={spec['args']}")
    return "\n".join(lines)

def llm_plan(user_goal: str, transcript: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Ask the LLM for the next step (tool call or final).
    transcript: list of {"role":"tool","name":..,"result":..} chunks so far.
    """
    tool_list = tool_catalog_markdown()
    messages = [
        {"role":"system","content":PLANNER_SYS},
        {"role":"user","content": f"User goal: {user_goal}\n\nAvailable tools:\n{tool_list}\n\nSo far:\n{json.dumps(transcript, ensure_ascii=False)}\n"}
    ]
    resp = oai.chat.completions.create(model=OPENAI_MODEL, messages=messages, temperature=0)
    raw = resp.choices[0].message.content.strip()
    # Robust JSON parse
    try:
        return json.loads(raw)
    except Exception:
        # try to find a JSON object in text
        start = raw.find("{"); end = raw.rfind("}")
        if start>=0 and end>=0 and end>start:
            return json.loads(raw[start:end+1])
        raise ValueError(f"Planner returned non-JSON:\n{raw}")


In [8]:
async def call_tool(client: MCPClient, name: str, args: Dict[str, Any]) -> Dict[str, Any]:
    res = await client.call_tool(name, args or {})
    return unwrap(res)

async def run_agent(user_goal: str, max_steps: int = 4) -> Dict[str, Any]:
    """
    Small agent loop:
    - Plan with LLM
    - If tool requested → call it → append result → loop
    - If final → return final answer
    """
    client = await mcp_connect(SERVER_URL)
    try:
        steps = []
        for i in range(max_steps):
            plan = llm_plan(user_goal, steps)
            action = plan.get("action","").lower()

            if action == "final":
                return {"answer": plan.get("answer",""), "steps": steps}

            if action == "tool":
                name = plan.get("name")
                args = plan.get("args") or {}
                if name not in TOOL_SPECS:
                    # if LLM picked a non-existent tool, fail gracefully back to planner
                    steps.append({"role":"tool","name":name,"error":"unknown tool", "args":args})
                    continue
                try:
                    result = await call_tool(client, name, args)
                    steps.append({"role":"tool","name":name,"args":args,"result":result})
                except Exception as e:
                    steps.append({"role":"tool","name":name,"args":args,"error":str(e)})
                    # planner will see the error and (hopefully) pivot
                    continue
            else:
                # invalid action; ask again
                steps.append({"role":"tool","name":"(none)","error":"invalid planner action"})
                continue

        # if loop ended without "final", ask for a wrap-up
        wrap = oai.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role":"system","content":"Write a short, direct answer using the provided tool results. If uncertain, say what is missing."},
                {"role":"user","content": json.dumps(steps, ensure_ascii=False)}
            ],
            temperature=0
        )
        return {"answer": wrap.choices[0].message.content.strip(), "steps": steps}

    finally:
        await mcp_close(client)


In [9]:
# Example 1: simple metadata task
result = asyncio.run(run_agent("List all students and show emails for those in the Computer department."))
print("ANSWER:\n", result["answer"], "\n")
print("DEBUG STEPS:"); pjson(result["steps"])


ANSWER:
 Students in the Computer department with their emails:
1. Ahmed Hassan - ahmed.hassan@university.edu
2. Aisha Khan - aisha.khan@university.edu 

DEBUG STEPS:
[
  {
    "role": "tool",
    "name": "list_students",
    "args": {},
    "result": {
      "students": [
        "Ahmed Hassan",
        "Aisha Khan",
        "David Miller",
        "Kenji Tanaka",
        "Lisa Johnson",
        "Maria Gonzalez",
        "Mark Thompson",
        "Priya Patel",
        "Rahul Sharma",
        "Sofia Rodriguez"
      ],
      "count": 10,
      "_receipt": {
        "tool_used": true,
        "server_time": "2025-11-03T07:12:20.240955+00:00",
        "request_id": "e5b8fb69-21c3-436e-b833-1bd2071e8ef4",
        "data_root": "/app/TRAINING DATA"
      }
    }
  },
  {
    "role": "tool",
    "name": "search_student_by_field",
    "args": {
      "field": "department",
      "value": "Computer"
    },
    "result": {
      "matches": [
        {
          "name": "Ahmed Hassan",
         

In [10]:
# Example 2: mix metadata + RAG
result = asyncio.run(run_agent(
    "Provide a small summary on architecture of learning platform project"
))
print("ANSWER:\n", result["answer"], "\n")
print("DEBUG STEPS:"); pjson(result["steps"])


ANSWER:
 The architecture of the learning platform project involves a decoupled, client-server design aimed at high scalability and resilience. The client-side is developed using a modern cross-platform framework, allowing deployment as a native mobile application. Key features include a mobile-first interface design optimized for small touchscreens, local storage using databases like IndexedDB or SQLite to store content and track interactions, and a synchronization service that manages offline events and synchronizes them when network availability is detected. 

DEBUG STEPS:
[
  {
    "role": "tool",
    "name": "search_rag",
    "args": {
      "query": "architecture of learning platform project",
      "top_k": 1
    },
    "result": {
      "query": "architecture of learning platform project",
      "model": "text-embedding-3-small",
      "top_k": 1,
      "results": [
        {
          "id": "d23cea9d-3395-48ff-bc1b-06be22084fd0",
          "score": 0.580697954,
          "meta

In [11]:
# Example 2: mix metadata + RAG
result = asyncio.run(run_agent(
    "Find me a student who can collaborate on a robtic project along with his email id "
))
print("ANSWER:\n", result["answer"], "\n")
print("DEBUG STEPS:"); pjson(result["steps"])


ANSWER:
 Kenji Tanaka is a student who can collaborate on a robotics project. His email is kenji.tanaka@university.edu. 

DEBUG STEPS:
[
  {
    "role": "tool",
    "name": "find_candidates",
    "args": {
      "topics": "robotics",
      "n": 1
    },
    "result": {
      "requested_topics": [
        "robotics"
      ],
      "candidates": [
        {
          "student": "Kenji Tanaka",
          "email": "kenji.tanaka@university.edu",
          "project_title": "Autonomous Delivery Robot with Obstacle Avoidance",
          "metadata": {
            "student_id": "STU2024004",
            "name": "Kenji Tanaka",
            "email": "kenji.tanaka@university.edu",
            "department": "Robotics Engineering",
            "project_title": "Autonomous Delivery Robot with Obstacle Avoidance",
            "submission_date": "2024-01-22",
            "academic_year": "Second Year",
            "supervisor": "Dr. Emily Zhang",
            "keywords": [
              "robotics",
     